# LeNet-5 — Gradient-Based Learning Applied to Document Recognition (1998)

_Papers · Computer Vision_

**The convolutional network that learned to read digits from raw pixels — convolution, subsampling, then classification, all trained by back-propagation.**

---

This notebook is a practice scaffold for the **LeNet-5 — Gradient-Based Learning Applied to Document Recognition (1998)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torchvision, torchvision.transforms as T

torch.manual_seed(0)

# --- 0. Worked example: track the spatial size through the conv/pool stack (32x32 in). ---
def conv_out(n, k): return n - k + 1          # valid conv, stride 1, no padding
def pool_out(n):    return n // 2             # 2x2 average pool, stride 2
n = 32
for name, f in [("C1 conv5", lambda n: conv_out(n,5)), ("S2 pool2", pool_out),
                ("C3 conv5", lambda n: conv_out(n,5)), ("S4 pool2", pool_out),
                ("C5 conv5", lambda n: conv_out(n,5))]:
    n = f(n); print(f"  {name}: side -> {n}")
# -> 28, 14, 10, 5, 1  (matches Fig. 2)


# --- 1. LeNet-5, layer for layer (paper Section II-B). ---
class LeNet5(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(1,  6, kernel_size=5)   # 32 -> 28,  6 feature maps
        self.s2 = nn.AvgPool2d(2, 2)                # 28 -> 14   (sub-sampling)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)   # 14 -> 10, 16 feature maps
        self.s4 = nn.AvgPool2d(2, 2)                # 10 -> 5
        self.c5 = nn.Conv2d(16, 120, kernel_size=5) # 5  -> 1, 120 maps (a full connection)
        self.f6  = nn.Linear(120, 84)               # fully-connected, 84 units
        self.out = nn.Linear(84, n_classes)         # 10 classes (linear head replaces RBF)

    def forward(self, x, trace=False):
        x = torch.tanh(self.c1(x));  s = [("C1", x.shape)]
        x = self.s2(x);              s += [("S2", x.shape)]
        x = torch.tanh(self.c3(x));  s += [("C3", x.shape)]
        x = self.s4(x);              s += [("S4", x.shape)]
        x = torch.tanh(self.c5(x));  s += [("C5", x.shape)]
        x = x.flatten(1)
        x = torch.tanh(self.f6(x))
        x = self.out(x)
        if trace:
            for name, sh in s: print(f"  {name}: {tuple(sh)}")
        return x

net = LeNet5()
print("\nReal tensor shapes through LeNet-5 (batch of 2, 1x32x32):")
_ = net(torch.zeros(2, 1, 32, 32), trace=True)   # -> 28,14,10,5,1 in the spatial dims
print("LeNet-5 free parameters:", sum(p.numel() for p in net.parameters()))


# --- 2. MNIST, resized to 32x32 (torchvision is preinstalled in Colab). ---
tfm = T.Compose([T.Resize((32, 32)), T.ToTensor(),
                 T.Normalize((0.1307,), (0.3081,))])
train = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tfm)
test  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tfm)
train = torch.utils.data.Subset(train, range(8000))   # small + fast
test  = torch.utils.data.Subset(test,  range(2000))
trl = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
tel = torch.utils.data.DataLoader(test,  batch_size=256)
device = "cuda" if torch.cuda.is_available() else "cpu"


def accuracy(model):
    model.eval(); correct = tot = 0
    with torch.no_grad():
        for xb, yb in tel:
            correct += (model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
            tot += len(yb)
    return correct / tot


def train_model(model, epochs=5, lr=0.1):
    model = model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    lf  = nn.CrossEntropyLoss()
    accs = []
    for ep in range(epochs):
        model.train()
        for xb, yb in trl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = lf(model(xb), yb); loss.backward(); opt.step()
        accs.append(accuracy(model))
        print(f"  epoch {ep}  test acc {accs[-1]:.4f}")
    return accs


# --- 3. Train LeNet-5 and print test accuracy. ---
print("\nLeNet-5 (convolutional):")
torch.manual_seed(0)
lenet_acc = train_model(LeNet5())

# --- 4. ABLATION: a parameter-matched fully-connected net on the SAME data. ---
class MLP(nn.Module):   # flatten 32x32 -> one hidden layer -> 10  (no convolution / no weight-sharing)
    def __init__(self, hidden=60):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(32*32, hidden),
                                 nn.Tanh(), nn.Linear(hidden, 10))
    def forward(self, x): return self.net(x)

print("\nFully-connected ablation (no convolution):")
torch.manual_seed(0)
mlp_acc = train_model(MLP())

print("\nFinal test accuracy  LeNet-5:", round(lenet_acc[-1], 4),
      " | MLP ablation:", round(mlp_acc[-1], 4))
# Our small run (8000-image subset, 5 epochs): LeNet-5 ~0.95, the MLP ablation lags.
# This is OUR small run, not the paper's reported 0.95% test error on full MNIST.

## Visualize the data & results

_On a small MNIST subset, does convolutional LeNet-5 generalize better than a parameter-matched fully-connected net (the ablation)?_

In [ ]:
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T
torch.manual_seed(0)

# Same two models as the CODE cell, compared on a small MNIST subset. The gap
# is the qualitative effect: convolution's weight-sharing generalizes from less data.
tfm = T.Compose([T.Resize((32,32)), T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
tr  = torch.utils.data.Subset(torchvision.datasets.MNIST(".",True ,download=True,transform=tfm), range(8000))
te  = torch.utils.data.Subset(torchvision.datasets.MNIST(".",False,download=True,transform=tfm), range(2000))
trl = torch.utils.data.DataLoader(tr, batch_size=128, shuffle=True)
tel = torch.utils.data.DataLoader(te, batch_size=256)

class LeNet5(nn.Module):
    def __init__(s):
        super().__init__()
        s.c1=nn.Conv2d(1,6,5); s.s2=nn.AvgPool2d(2,2)
        s.c3=nn.Conv2d(6,16,5);s.s4=nn.AvgPool2d(2,2)
        s.c5=nn.Conv2d(16,120,5); s.f6=nn.Linear(120,84); s.o=nn.Linear(84,10)
    def forward(s,x):
        x=s.s2(torch.tanh(s.c1(x))); x=s.s4(torch.tanh(s.c3(x)))
        x=torch.tanh(s.c5(x)).flatten(1); return s.o(torch.tanh(s.f6(x)))

class MLP(nn.Module):
    def __init__(s,h=60):
        super().__init__()
        s.n=nn.Sequential(nn.Flatten(),nn.Linear(1024,h),nn.Tanh(),nn.Linear(h,10))
    def forward(s,x): return s.n(x)

def run(model):
    opt=torch.optim.SGD(model.parameters(),lr=0.1,momentum=0.9); lf=nn.CrossEntropyLoss(); accs=[]
    for ep in range(5):
        model.train()
        for xb,yb in trl: opt.zero_grad(); lf(model(xb),yb).backward(); opt.step()
        model.eval(); c=t=0
        with torch.no_grad():
            for xb,yb in tel: c+=(model(xb).argmax(1)==yb).sum().item(); t+=len(yb)
        accs.append(round(c/t,4))
    return accs

torch.manual_seed(0); print("LeNet-5:", run(LeNet5()))
torch.manual_seed(0); print("MLP    :", run(MLP()))
# Our small run -> LeNet-5: [0.8805, 0.93, 0.942, 0.9515, 0.954]
#                  MLP    : [0.8765, 0.9075, 0.903, 0.9065, 0.914]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have LeNet-5 hitting good accuracy on a small MNIST subset. Replace the
            entire convolution/pooling stack with a plain fully-connected net (flatten the 32&times;32 image to
            1024 numbers &rarr; one hidden layer &rarr; 10 outputs) with a comparable parameter count, train on
            the same small subset, and compare test accuracy. What do you expect, and what does the
            result demonstrate about the paper's claim?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the FC baseline: Flatten &rarr; Linear(1024, H) &rarr; tanh &rarr; Linear(H, 10), picking H so the parameter count is in LeNet-5's ballpark. — _A fair ablation matches capacity, so any gap is due to the convolutional structure (weight-sharing + locality), not raw parameter count._
- Train both on the identical subset, epochs, and optimizer; compare final test accuracy. — _Holding everything else fixed isolates the architecture as the only variable._
- Observe the conv net generalizes better (higher test accuracy) on the small data. — _Weight-sharing bakes in the prior that a feature is useful anywhere in the image, so the conv net needs less data to learn shift-invariant digit features; the FC net must re-learn each feature at each position._

**Answer:** On the same small subset the convolutional LeNet-5 generalizes better than a
                 parameter-matched fully-connected net. Because both have similar capacity, the gap isolates the
                 convolutional structure &mdash; local receptive fields plus shared weights &mdash; as the
                 reason. That is exactly the paper's argument (&sect;II): for images, weight-sharing across
                 positions is the right prior, so the conv net learns shift-invariant features from less data.
                 The CODEVIZ panel shows the test-accuracy gap.

</details>

**Problem 2.** Trace the spatial size through LeNet-5 by hand for a 32&times;32 input, layer by layer, using
            $n\to n-4$ for each 5&times;5 conv and $n\to n/2$ for each 2&times;2 pool. What is the side length
            after C5, and why does that let C5 act as a "full connection"?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- C1: $32-4=28$. S2: $28/2=14$. C3: $14-4=10$. S4: $10/2=5$. C5: $5-4=1$. — _Each valid 5×5 conv removes 4 from the side; each 2×2 stride-2 pool halves it._
- Note S4's output is 5×5, and C5 uses a 5×5 filter. — _A filter the same size as its input has exactly one valid placement._
- Conclude each C5 unit sees the entire 5×5×16 block at once, so C5 is effectively fully-connected to S4. — _With a 1×1 output map, "sliding the filter" reduces to a single dot product over all inputs — a full connection._

**Answer:** The side length goes $32\to28\to14\to10\to5\to1$, so after C5 the map is 1&times;1.
                 Because S4 is 5&times;5 and C5's filter is also 5&times;5, the filter has just one valid
                 placement and each of the 120 C5 units reads the whole 5&times;5&times;16 input at once.
                 That makes C5 a full connection in convolution's clothing &mdash; which is exactly why the paper
                 says C5 is "labeled as a convolutional layer, instead of a fully-connected layer" only because a
                 larger input would make its output bigger than 1&times;1.

</details>

**Problem 3.** In the original OUTPUT layer, unit $i$ computes $y_i = \sum_j (x_j - w_{ij})^2$ over the 84
            F6 outputs $x$ and its prototype $w_i$. Suppose, with a toy 3-number F6 vector, $x=[1,0,1]$ and two
            class prototypes $w_0=[1,0,1]$, $w_1=[0,1,0]$. Compute $y_0$ and $y_1$. Which class does the network
            predict, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $y_0 = (1-1)^2+(0-0)^2+(1-1)^2 = 0$. — _Eqn. 7 is the squared Euclidean distance; a perfect match gives 0._
- $y_1 = (1-0)^2+(0-1)^2+(1-0)^2 = 1+1+1 = 3$. — _Each mismatched coordinate contributes its squared difference._
- Pick the class with the smallest $y_i$: $y_0=0 < y_1=3$, so predict class 0. — _A smaller RBF output means the F6 description is closer to that class's prototype._

**Answer:** $y_0 = 0$ and $y_1 = 3$. The network predicts class 0, because the RBF output is the
                 squared distance to each class prototype and the smallest distance wins. Here F6's
                 vector exactly equals prototype $w_0$ (distance 0) and is far from $w_1$ (distance 3). This is
                 why the paper calls a small $y_i$ a good "fit": the RBF output is "the unnormalized negative
                 log-likelihood of a Gaussian" centered on the prototype.

</details>